## 🎙️🪽 飛翔用のAI文字起こしツール

Google Colab上でAIモデルを動かし、音声ファイルの**文字起こし**と**話者分離（誰が話したか）** を行います。結果は時間&話者ラベル付きの `.txt` ファイルで保存します。

---

###前提事項: 詳しくはスライド参照
```
①自身のGoogle アカウントでログインしている。  
②共有のGoogle Driveフォルダへのアクセス（**編集権限**）があり、ショートカットを作成した。  
③文字起こししたい音声ファイルが上記/audioフォルダにアップロードされている。
```

###参考: 共有フォルダの中身
```
<[2026]飛翔AI文字起こし>/              ← 準備B の BASE_FOLDER_NAME と名称一致させる
├─ model-community-1/   ←モデルのキャッシュ用
├─ audio/               ←ここに音声ファイルを置く
└─ transcripts/         ←ここに結果が出力される
```

### このノートブックの使い方
1. このノートブックを自身のGoogleアカウントにログイン状態で開く
2. 上部メニューバーの **ランタイム → ランタイムのタイプを変更 → T4 GPU**
3. **準備A** を実行する（完了したら「ランタイム → セッションを再起動する」を実行する）
4. 再起動後 **準備B** を実行 → `BASE_FOLDER_NAME` は変更不要。ドライブ接続を求められたらすべて許可する。
5. **設定・音声選択・実行** セルで言語・話者数などを設定
6. `audio/` 内の任意の音声を選び **▶ 文字起こし開始** → `transcripts/` に保存される

In [ ]:
# @title ⚙️ 準備A：パッケージのインストール（まずここを実行） { display-mode: "form" }
import importlib.util, subprocess, sys, os

# whisperx 未導入なら GPU 確認 → インストール → 手動でセッション再起動
if importlib.util.find_spec('whisperx') is None:
    gpu = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
        capture_output=True, text=True)
    if gpu.returncode != 0:
        print('❌ GPUが検出されません。「ランタイムのタイプを変更」→ T4 GPU を選んでください。')
        raise SystemExit()
    print(f'✅ GPU: {gpu.stdout.strip()}')
    print('📦 パッケージをインストール中... しばらくお待ちください（数分）')

    # whisperx のみ導入し、numpy/torch などの依存は pip の解決に委ねる
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', 'whisperx']
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0 and 'Successfully installed' not in r.stdout:
        print('❌ whisperx のインストールに失敗:')
        print(r.stderr[-1500:])
        raise SystemExit()

    print('✅ インストール完了。')
    print('   メニューの「ランタイム → セッションを再起動する」を実行してください。')
    print('   再起動後、次の「準備B」セルへ進みます。')
    # 自動再起動(os._exit/os.kill)は使わない（コードからの終了はクラッシュ表記が出るため手動再起動にする）
else:
    print('✅ whisperx は導入済みです。次の「準備B」セルを実行してください。')

✅ GPU: Tesla T4, 15360 MiB
📦 パッケージをインストール中... しばらくお待ちください（3-4分程度）


In [ ]:
# @title ⚙️ 準備B：ドライブ接続の許可（準備A完了→再起動後に実行） { display-mode: "form" }
import os, subprocess
from google.colab import drive

gpu = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True)
gpu_info = gpu.stdout.strip() if gpu.returncode == 0 else '不明'

print(f'✅ GPU: {gpu_info}')
print('✅ パッケージ準備完了')
print('📂 Google Drive に接続中...')
drive.mount('/content/drive')

# ▼ 自分のDrive共有フォルダ名に合わせて変更してください
BASE_FOLDER_NAME = '[2026]飛翔AI文字起こし'  # @param {type:"string"}
cands = [
    f'/content/drive/MyDrive/{BASE_FOLDER_NAME}',
    f'/content/drive/My Drive/{BASE_FOLDER_NAME}',
]
mydrive = '/content/drive/MyDrive'
if os.path.isdir(mydrive):
    for d in os.listdir(mydrive):
        p = os.path.join(mydrive, d, BASE_FOLDER_NAME)
        if os.path.isdir(p):
            cands.append(p)

existing = [c for c in cands if os.path.isdir(c)]

def looks_like_base(p):
    try: return any(x in os.listdir(p) for x in ('model-community-1', 'audio', 'transcripts'))
    except Exception: return False

BASE_DIR = next((c for c in existing if looks_like_base(c)), existing[0] if existing else None)
if BASE_DIR is None:
    print(f'❌ 共有フォルダ「{BASE_FOLDER_NAME}」が見つかりません。')
    print('   ショートカットをマイドライブ直下に追加したか確認してください')
    raise SystemExit()

MODEL_DIR  = f'{BASE_DIR}/model-community-1'
AUDIO_DIR  = f'{BASE_DIR}/audio'
OUTPUT_DIR = f'{BASE_DIR}/transcripts'
DEVICE = 'cuda'; COMPUTE_TYPE = 'float16'; BATCH_SIZE = 16

model_ready = os.path.isdir(MODEL_DIR) and bool(os.listdir(MODEL_DIR))
print(f'✅ 共有フォルダ: {BASE_DIR}')
print(f'   モデル: {"準備済み ✅" if model_ready else "❌ 未準備 — オーナーにセットアップ版の実行を依頼してください"}')
print()
print('準備完了。次のセルを実行してください。')


In [ ]:
# @title 🎵 設定・音声選択・実行 { display-mode: "form" }
import os
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── 設定（フォームで変更できます）──
MODEL_SIZE     = 'large-v3'  # @param ["medium", "large-v3"]
LANGUAGE       = 'ja'        # @param ["ja", "en", "zh", "ko"]
NUM_SPEAKERS   = 0            # @param {type:"integer"}
INITIAL_PROMPT = ''           # @param {type:"string"}
# LANGUAGE: 音声の言語（ja=日本語, en=英語, zh=中国語, ko=韓国語）
# NUM_SPEAKERS: 0 = 自動検出、人数が分かっていれば整数（例: 3）
# INITIAL_PROMPT: 固有名詞があれば入力（例: 会社名・専門用語）


# ── 音声ファイルを列挙 ──
EXTS = ('.mp3', '.wav', '.m4a', '.mp4', '.ogg', '.flac')

def find_audio_dir(base):  # 'Audio' など大文字小文字ゆれに対応
    cand = os.path.join(base, 'audio')
    if os.path.isdir(cand): return cand
    for d in (os.listdir(base) if os.path.isdir(base) else []):
        if d.lower() == 'audio' and os.path.isdir(os.path.join(base, d)):
            return os.path.join(base, d)
    return cand

ADIR = find_audio_dir(BASE_DIR)
files = sorted(
    os.path.join(ADIR, name)
    for name in (os.listdir(ADIR) if os.path.isdir(ADIR) else [])
    if name.lower().endswith(EXTS) and os.path.isfile(os.path.join(ADIR, name))
)
if not files:
    print(f'❌ 音声が見つかりません: {ADIR}')
    print('   mp3/wav/m4a/mp4/ogg/flac を audio/ フォルダに置いてください')
    raise SystemExit()

# ── ウィジェット ──
audio_dropdown = widgets.Dropdown(
    options=[(os.path.basename(f), f) for f in files],
    description='音声:',
    layout=widgets.Layout(width='80%'),
    style={'description_width': 'initial'}
)
run_button = widgets.Button(
    description='▶ 文字起こし開始',
    button_style='success',
    layout=widgets.Layout(width='200px', height='40px')
)
output_area = widgets.Output()

# ── 実行ハンドラ（重いインポートはここで遅延ロード）──
def on_run(b):
    run_button.disabled = True
    with output_area:
        clear_output(wait=True)
        import gc, time, shutil, datetime, torch, whisperx, pandas as pd
        from pathlib import Path
        from pyannote.audio import Pipeline

        # 大きいオブジェクトは None 化して gc → VRAM解放（参照が確実に切れる）
        model = model_a = pipeline = None

        def free():
            gc.collect()
            try: torch.cuda.empty_cache()
            except Exception: pass

        def fmt_time(t):
            h, rem = divmod(int(t), 3600); m, s = divmod(rem, 60)
            return f'{h:02d}:{m:02d}:{s:02d}' if h else f'{m:02d}:{s:02d}'

        def to_srt_time(t):
            h, rem = divmod(t, 3600); m, s = divmod(rem, 60)
            return f'{int(h):02d}:{int(m):02d}:{int(s):02d},{int((s % 1) * 1000):03d}'

        def render_segments(segments):
            lines, cur = [], None
            for seg in segments:
                sp = seg.get('speaker', 'SPEAKER_?'); txt = seg['text'].strip()
                if sp != cur: lines.append(f'\n─── {sp} ───'); cur = sp
                lines.append(f'[{fmt_time(seg["start"])}〜{fmt_time(seg["end"])}] {txt}')
            return lines

        def load_local_pipeline(model_dir):
            # ローカル（Drive）のモデルをオフライン読込み
            os.environ['HF_HUB_OFFLINE'] = '1'
            try:
                return Pipeline.from_pretrained(model_dir)
            finally:
                os.environ.pop('HF_HUB_OFFLINE', None)

        try:
            _num_speakers = NUM_SPEAKERS if NUM_SPEAKERS > 0 else None
            SRC = audio_dropdown.value
            AUDIO_FILE = f'/content/{os.path.basename(SRC)}'
            shutil.copy(SRC, AUDIO_FILE)
            print(f'🎵 対象: {os.path.basename(SRC)}')

            print('🔄 [1/4] 音声を読み込み中...')
            audio = whisperx.load_audio(AUDIO_FILE)
            print(f'   長さ: {len(audio) / 16000 / 60:.1f}分')

            print(f'🔄 [2/4] モデル({MODEL_SIZE})で文字起こし中...')
            t0 = time.time()
            asr_options = {'initial_prompt': INITIAL_PROMPT} if INITIAL_PROMPT else None
            model = whisperx.load_model(MODEL_SIZE, DEVICE, compute_type=COMPUTE_TYPE,
                                        language=LANGUAGE, asr_options=asr_options)
            result = model.transcribe(audio, batch_size=BATCH_SIZE, language=LANGUAGE)
            print(f'   完了（{time.time() - t0:.0f}秒）/ {len(result["segments"])}セグメント')
            model = None; free()

            print('🔄 [3/4] タイムスタンプを精緻化中...')
            try:
                model_a, metadata = whisperx.load_align_model(language_code=LANGUAGE, device=DEVICE)
                result = whisperx.align(result['segments'], model_a, metadata, audio, DEVICE,
                                        return_char_alignments=False)
                model_a = None; free()
                print('   完了')
            except Exception as e:
                print(f'   ⚠️ 精緻化をスキップ（{e}）')
                print('      → 単語境界が粗くなり、話者ラベルがずれる場合があります')

            print('🔄 [4/4] 話者を識別中...')
            try:
                if not (os.path.isdir(MODEL_DIR) and os.listdir(MODEL_DIR)):
                    raise FileNotFoundError('モデル未準備')
                pipeline = load_local_pipeline(MODEL_DIR)
                pipeline.to(torch.device(DEVICE))
                waveform = torch.from_numpy(audio).unsqueeze(0)
                pk = {'num_speakers': _num_speakers} if _num_speakers else {}
                out = pipeline({'waveform': waveform, 'sample_rate': 16000}, **pk)
                annotation = getattr(out, 'speaker_diarization', out)
                diarize_df = pd.DataFrame(annotation.itertracks(yield_label=True),
                                          columns=['segment', 'label', 'speaker'])
                diarize_df['start'] = diarize_df['segment'].apply(lambda x: x.start)
                diarize_df['end']   = diarize_df['segment'].apply(lambda x: x.end)
                result = whisperx.assign_word_speakers(diarize_df, result)
                speakers = sorted({s['speaker'] for s in result['segments'] if 'speaker' in s})
                print(f'   完了: {len(speakers)}人 ({", ".join(speakers)})')
            except Exception as e:
                print(f'   ⚠️ 話者分離をスキップ（{type(e).__name__}: {e}）')
            finally:
                pipeline = None; free()

            stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
            base  = f'{Path(AUDIO_FILE).stem}_{stamp}'

            def build_txt():
                header = ['# 文字起こし結果', f'# 音声: {Path(AUDIO_FILE).name}', '']
                return '\n'.join(header + render_segments(result['segments']))

            def build_srt():
                out = []
                for i, seg in enumerate(result['segments'], 1):
                    sp = seg.get('speaker', '??'); txt = seg['text'].strip()
                    out.append(f'{i}\n{to_srt_time(seg["start"])} --> {to_srt_time(seg["end"])}\n[{sp}] {txt}\n')
                return '\n'.join(out)

            txt_data, srt_data = build_txt(), build_srt()

            print('\n' + '=' * 60)
            print('【プレビュー】')
            print('=' * 60)
            print('\n'.join(render_segments(result['segments'])))

            try:
                os.makedirs(OUTPUT_DIR, exist_ok=True)
                for path, data in [(f'{OUTPUT_DIR}/{base}.txt', txt_data),
                                   (f'{OUTPUT_DIR}/{base}.srt', srt_data)]:
                    with open(path, 'w', encoding='utf-8') as f: f.write(data)
                print(f'\n✅ 保存完了: {OUTPUT_DIR}/{base}.txt / .srt')
            except Exception as e:
                print(f'\n⚠️ Drive保存に失敗（{type(e).__name__}）。端末ダウンロードに切替えます。')
                from google.colab import files as colab_files
                for name, data in [(f'{base}.txt', txt_data), (f'{base}.srt', srt_data)]:
                    local = f'/content/{name}'
                    with open(local, 'w', encoding='utf-8') as f: f.write(data)
                    colab_files.download(local)

        except Exception as e:
            import traceback
            print(f'❌ エラー: {type(e).__name__}: {e}')
            traceback.print_exc()
        finally:
            model = model_a = pipeline = None; free()
            run_button.disabled = False

run_button.on_click(on_run)
display(audio_dropdown, run_button, output_area)